In [1]:
!wget https://s3.amazonaws.com/tripdata/202412-citibike-tripdata.zip

--2025-01-18 17:01:20--  https://s3.amazonaws.com/tripdata/202412-citibike-tripdata.zip
Resolving s3.amazonaws.com (s3.amazonaws.com)... 54.231.193.56, 16.182.73.216, 52.216.58.64, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|54.231.193.56|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 450888449 (430M) [application/zip]
Saving to: ‘202412-citibike-tripdata.zip’

202412-citibike-tri 100%[===================>] 430.00M  47.3MB/s    in 10s     

2025-01-18 17:01:30 (42.4 MB/s) - ‘202412-citibike-tripdata.zip’ saved [450888449/450888449]



In [2]:
!unzip /content/202412-citibike-tripdata.zip

Archive:  /content/202412-citibike-tripdata.zip
 extracting: 202412-citibike-tripdata_1.csv  
 extracting: 202412-citibike-tripdata_3.csv  
 extracting: 202412-citibike-tripdata_2.csv  


In [3]:
!pip install pandera -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.4/261.4 kB 7.8 MB/s eta 0:00:00


In [4]:
!rm -rf gx
!pip install pandas
!pip install great_expectations -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.6/813.6 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.1/739.1 kB 24.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.1.4 which is incompatible.
mizani 0.13.1 requires pandas>=2.2.0, but you have pandas 2.1.4 which is incompatible.
plotnine 0.14.5 requires pandas>=2.2.0, but you have pandas 2.1.4 which is incompatible.


In [5]:
# Importing necessary libraries and modules
import warnings
import json
import pandas as pd
import pandera as pa
import great_expectations as gx
from great_expectations.data_context.types.base import DataContextConfig
from great_expectations.checkpoint import CheckpointResult, SlackNotificationAction, UpdateDataDocsAction, EmailAction


In [6]:
# Load the CSV file into a DataFrame
file_path = "202412-citibike-tripdata_1.csv"
df = pd.read_csv(file_path)
df.sample(5) # Observing the 5 sample rows

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
<ipython-input-6-046a8565ea5b>:3: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
79151,20EA1F279032F584,electric_bike,2024-12-03 07:51:01.531,2024-12-03 08:02:01.668,W 43 St & 10 Ave,6756.01,Broadway & W 25 St,6173.08,40.760094,-73.994618,40.742869,-73.989186,member
443968,EAE543D0CFBC4612,classic_bike,2024-12-04 16:50:26.438,2024-12-04 17:27:47.976,8 Ave & W 38 St,6526.05,Riverside Dr & W 138 St,7942.21,40.754610,-73.991770,40.823168,-73.955857,casual
661913,846BBABD9994ACFB,electric_bike,2024-12-12 18:41:25.975,2024-12-12 18:44:43.266,7 Ave & W 55 St,6847.05,W 56 St & 10 Ave,6955.01,40.764126,-73.980973,40.768254,-73.988639,member
796005,C0C8A681BFB0C240,classic_bike,2024-12-10 17:15:04.863,2024-12-10 17:56:14.609,1 Ave & E 44 St,6379.03,31 Ave & 57 St,6621.06,40.750020,-73.969053,40.757357,-73.904726,member
880597,1B0BF9BF8848B3FD,classic_bike,2024-12-05 22:07:01.761,2024-12-05 22:10:46.107,W 51 St & 6 Ave,6740.10,8 Ave & W 52 St,6816.07,40.760660,-73.980420,40.763707,-73.985162,member


# Question 1 : Pandera Validation Rules

## a. Validate data type for each column

In [7]:
# Define schema for the dataframe
schema1 = pa.DataFrameSchema({
    "ride_id": pa.Column(str, nullable=True), # Data type must be str
    "rideable_type": pa.Column(str, nullable=True),
    "started_at": pa.Column(str, nullable=True),
    "ended_at": pa.Column(str, nullable=True),
    "start_station_name": pa.Column(str, nullable=True),
    "start_station_id": pa.Column(str, nullable=True),
    "end_station_name": pa.Column(str, nullable=True),
    "end_station_id": pa.Column(str, nullable=True),
    "start_lat": pa.Column(float,  nullable=True), # Data type must be float
    "start_lng": pa.Column(float,  nullable=True),
    "end_lat": pa.Column(float,  nullable=True),
    "end_lng": pa.Column(float,  nullable=True),
    "member_casual": pa.Column(str,nullable=True), # Data type must be str
}, strict=True)

# Validate the dataframe
try:
    schema1.validate(df, lazy=True) # Setting lazy as True collects all validation errors instead of stopping at the first error
    print("Data validated successfully!")
except pa.errors.SchemaErrors as e:
    print("Validation errors occurred:")
    print(json.dumps(e.message, indent=2)) # Errors are formatted into a JSON-like string

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.11/dist-packages/pyspark/sql/pandas/utils.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pandas.__version__) < LooseVersion(minimum_pandas_version):
/usr/local/lib/python3.11/dist-packages/pyspark/sql/pandas/utils.py:64: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pyarrow.__version__) < LooseVersion(minimum_pyarrow_version):
/usr/local/lib/python3.11/dist-packages/pyspark/pandas/__init__.py:47: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version in

Validation errors occurred:
{
  "SCHEMA": {
    "WRONG_DATATYPE": [
      {
        "schema": null,
        "column": "start_station_id",
        "check": "dtype('str')",
        "error": "expected series 'start_station_id' to have type str:failure cases:        index failure_case0      983040      6257.061      983041      6667.042      983042      6847.023      983043      3919.074      983044      3919.07...       ...          ...16949  999995      6248.0816950  999996      5422.0916951  999997      7655.2216952  999998      5578.0216953  999999      8277.03[16954 rows x 2 columns]"
      },
      {
        "schema": null,
        "column": "end_station_id",
        "check": "dtype('str')",
        "error": "expected series 'end_station_id' to have type str:failure cases:         index failure_case0       262144      5553.031       262145      5838.092       262146      6123.023       262147      6046.024       262148      6046.02...        ...          ...145407  999995      6459.0

## b. Write appropriate check for each feature

In [8]:
# Define schema for the dataframe
schema2 = pa.DataFrameSchema({
    "ride_id": pa.Column(str, checks=[pa.Check(lambda x: x.str.len().nunique() == 1), pa.Check.str_length(1, None),], nullable=True), # Each ride_id must have a consistent length
    "rideable_type": pa.Column(str, checks=pa.Check.isin(["classic_bike", "electric_bike"]), nullable=True), # Must be one of two categories: "classic_bike" or "electric_bike"
    "started_at": pa.Column(str, nullable=True), # Allow null values
    "ended_at": pa.Column(str, nullable=True),
    "start_station_name": pa.Column(str, nullable=True),
    "start_station_id": pa.Column(str, nullable=True),
    "end_station_name": pa.Column(str, nullable=True),
    "end_station_id": pa.Column(str, nullable=True),
    "start_lat": pa.Column(float, checks=pa.Check(lambda x: (-90 <= x) & (x <= 90)), nullable=True), # Latitude (start_lat, end_lat) must be between -90 and 90 degrees.
    "start_lng": pa.Column(float, checks=pa.Check(lambda x: (-180 <= x) & (x <= 180)), nullable=True), # Longitude (start_lng, end_lng) must be between -180 and 180 degrees.
    "end_lat": pa.Column(float, checks=pa.Check(lambda x: (-90 <= x) & (x <= 90)), nullable=True),
    "end_lng": pa.Column(float, checks=pa.Check(lambda x: (-180 <= x) & (x <= 180)), nullable=True),
    "member_casual": pa.Column(str, checks=pa.Check.isin(["member", "casual"]), nullable=True) # Must be either "member" or "casual"
}, strict=True)

# Validate the dataframe
try:
    schema2.validate(df, lazy=True)
    print("Data validated successfully!")
except pa.errors.SchemaErrors as e:
    print("Validation errors occurred:")

    print(json.dumps(e.message, indent=2))


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Validation errors occurred:
{
  "SCHEMA": {
    "WRONG_DATATYPE": [
      {
        "schema": null,
        "column": "start_station_id",
        "check": "dtype('str')",
        "error": "expected series 'start_station_id' to have type str:failure cases:        index failure_case0      983040      6257.061      983041      6667.042      983042      6847.023      983043      3919.074      983044      3919.07...       ...          ...16949  999995      6248.0816950  999996      5422.0916951  999997      7655.2216952  999998      5578.0216953  999999      8277.03[16954 rows x 2 columns]"
      },
      {
        "schema": null,
        "column": "end_station_id",
        "check": "dtype('str')",
        "error": "expected series 'end_station_id' to have type str:failure cases:         index failure_case0       262144      5553.031       262145      5838.092       262146      6123.023       262147      6046.024       262148      6046.02...        ...          ...145407  999995      6459.0

## c. Start and end time validation

In [9]:
# Defining the schema
schema = pa.DataFrameSchema(
    columns={
        "started_at": pa.Column(pd.Timestamp, nullable=True),
        "ended_at": pa.Column(pd.Timestamp, nullable=True),
    },
    checks=pa.Check(lambda df: (df["started_at"] < df["ended_at"]))
)

# Conversion to date-time format
df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])

# Decorator for validation
@pa.check_input(schema)
def validate_dataframe(df):
    # Validate the dataframe
    try:
        schema.validate(df, lazy=True)
        print("Data validated successfully!")
    except pa.errors.SchemaErrors as e:
        print("Validation errors occurred:")
        # Print validation errors formatted as JSON
        print(json.dumps(e.message, indent=2))

validate_dataframe(df)

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Data validated successfully!


# Question 2: Greate Expectation Suite

## a. Validate data type for each column

In [10]:
#1. Create a Data Context
context = gx.get_context(mode="file")
print(type(context).__name__)

FileDataContext


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [11]:
# 2. Set up a Datasource (CSV file example)
data_source_name = "tripdata"
data_source = context.data_sources.add_pandas(name=data_source_name)

# A dataframe Data Asset is used to group our Validation Results.
data_asset_name = "tripdata_data_asset"
data_asset = data_source.add_dataframe_asset(name=data_asset_name)

In [12]:
# 3. Create a Batch from the DataFrame

# Batch Definitions for dataframe Data Assets don't work to subdivide
# the data returned for validation. Instead, they serve as an additional layer of
# organization and allow us to further group our Validation Results.

batch_definition = data_asset.add_batch_definition_whole_dataframe("batch definition")
batch_parameters = {"dataframe": df}
batch = batch_definition.get_batch(batch_parameters)
print(batch.head(3))

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

            ride_id  rideable_type              started_at  \
0  B44E5B10AEE58AD0   classic_bike 2024-12-14 10:58:18.153   
1  BC252DC6A6011556  electric_bike 2024-12-12 14:46:12.473   
2  6FBE55EF6FE8736D  electric_bike 2024-12-11 07:55:18.770   

                 ended_at                  start_station_name  \
0 2024-12-14 11:11:11.308  Frederick Douglass Blvd & W 145 St   
1 2024-12-12 16:45:37.777               Madison Ave & E 99 St   
2 2024-12-11 08:02:23.460               Columbia St & Kane St   

  start_station_id  end_station_name end_station_id  start_lat  start_lng  \
0          7954.12  E 138 St & 5 Ave        7809.13  40.823061 -73.941928   
1          7443.01               NaN            NaN  40.789485 -73.952429   
2          4422.05               NaN            NaN  40.687632 -74.001626   

    end_lat    end_lng member_casual  
0  40.81449 -73.936153        member  
1  40.78000 -73.960000        member  
2  40.69000 -74.000000        member  


In [13]:
# 4. Create an Expectation Suite
suite_name = "tripdata_expectation_suite"
suite = gx.ExpectationSuite(name=suite_name)
suite = context.suites.add(suite)

# Add expectations to the suite
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column="ride_id", type_="str") # Data type must be str
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column="rideable_type", type_="str")
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column="started_at", type_="str")
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column="ended_at", type_="str")
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column="start_station_name", type_="str")
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column="start_station_id", type_="str")
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column="end_station_name", type_="str")
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column="end_station_id", type_="str")
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column="start_lat", type_="float") # Data type must be float
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column="start_lng", type_="float")
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column="end_lat", type_="float")
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column="end_lng", type_="float")
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column="member_casual", type_="str") # Data type must be str
)


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


ExpectColumnValuesToBeOfType(id='33577044-e030-4f16-9355-078bc839eb8f', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, windows=None, batch_id=None, column='member_casual', mostly=1, row_condition=None, condition_parser=None, type_='str')

In [14]:
# Define and save the ValidationDefinition
definition_name = "tripdata_definition"
validation_definition = gx.ValidationDefinition(
    data=batch_definition, suite=suite, name=definition_name
)
context.validation_definitions.add(validation_definition)

# Run the validation using the saved ValidationDefinition
validation_results = validation_definition.run(batch_parameters=batch_parameters)
print(validation_results)

Calculating Metrics:   0%|          | 0/52 [00:00<?, ?it/s]

{
  "success": false,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_be_of_type",
        "kwargs": {
          "batch_id": "tripdata-tripdata_data_asset",
          "column": "ride_id",
          "type_": "str"
        },
        "meta": {},
        "id": "f117a264-2c61-4fa8-a54a-f5eb6fcfb8cc"
      },
      "result": {
        "element_count": 1000000,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
        "missing_count": 0,
        "missing_percent": 0.0,
        "unexpected_percent_total": 0.0,
        "unexpected_percent_nonmissing": 0.0,
        "partial_unexpected_counts": [],
        "partial_unexpected_index_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": true,
      "expectation_config": {
     

## b. Send a mail in case of failure

In [15]:
# Get the Great Expectations context
context = gx.get_context()

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [16]:
# Retrieve the validation definition by name
validation_definitions = [
    context.validation_definitions.get("tripdata_definition")
]

In [17]:
from google.colab import userdata
email_passowrd = userdata.get('EMAIL_PASSWORD')

In [18]:
# Define an email notification action to send alerts on validation results
email_action = [
    EmailAction(
        name="send_email_on_failure",  # Name of the action
        smtp_address='smtp.gmail.com',  # SMTP server address for sending emails
        smtp_port=587,  # SMTP server port
        receiver_emails="adiborate23@gmail.com",  # Email recipient
        sender_login="23110065@iitgn.ac.in",  # Email sender's login
        sender_password=email_passowrd,  # Email sender's password (consider using a secure method for this)
        use_tls=True,  # Use TLS for secure email transmission
        notify_on="all",  # Trigger notification on all validation outcomes
    )
]

In [19]:
checkpoint_name = "my_checkpoint"
# Create the checkpoint
checkpoint = gx.Checkpoint(
    name=checkpoint_name,
    validation_definitions=validation_definitions,
    actions=email_action,
    result_format={"result_format": "COMPLETE"},
)

# Save the Checkpoint to the Data Context
context.checkpoints.add(checkpoint)

# Retrieve the Checkpoint later
checkpoint_name = "my_checkpoint"
checkpoint = context.checkpoints.get(checkpoint_name)

In [20]:
# Run the checkpoint
validation_results = checkpoint.run(
    batch_parameters=batch_parameters, expectation_parameters=suite
)

Calculating Metrics:   0%|          | 0/52 [00:00<?, ?it/s]